# RT Notebook 26 — Authoritative D/E Clean-Room Candidate Comparison

This notebook is the governed successor to the stand-in harness run.

It will **not** execute an authoritative comparison unless all of the following are supplied:

1. the exact frozen governed candidate source,
2. the declared candidate SHA-256,
3. the governed candidate implementation ID,
4. clean-room reviewer sign-off,
5. the frozen independent formal-oracle specification.

The notebook:

- verifies candidate and specification hashes,
- imports the candidate through an isolated adapter contract,
- embeds the clean-room oracle independently,
- generates a new frozen corpus,
- compares all declared outputs,
- performs deterministic replay,
- preserves every disagreement,
- emits `RESULTS_002`,
- refuses to issue `PASS_BOUNDED_EQUIVALENCE` when governance prerequisites are absent.

## Candidate adapter contract

The supplied candidate Python file must export:

```python
candidate_representable(record, threshold_environment) -> str
candidate_noncollapsed(record, threshold_environment) -> str
candidate_admissible(record, threshold_environment) -> bool
```

Expected rejection/result strings:

```text
REJECT_TYPE
REJECT_CONTEXT
REJECT_PROFILE
REJECT_WITNESS
REJECT_HISTORY
REPRESENTABLE
REJECT_DISTINCTION
REJECT_SUBTHRESHOLD
NON_COLLAPSED
```


In [ ]:

from pathlib import Path
import json, hashlib, random, copy, zipfile, platform, sys, importlib.util
from datetime import datetime, timezone

ROOT = Path.cwd()
NOTEBOOK_ID = "RT_NOTEBOOK_26_D_E_CLEANROOM_CANDIDATE_COMPARISON_001"
RESULT_ID = "RT_NOTEBOOK_26_D_E_CLEANROOM_CANDIDATE_COMPARISON_001_RESULTS_002"
OUT = ROOT / RESULT_ID
OUT.mkdir(parents=True, exist_ok=True)

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(canonical(row) + "\n")

print("Output directory:", OUT.resolve())


## Governed execution inputs

Set these values before execution. The notebook deliberately stops when placeholders remain.


In [ ]:

# REQUIRED GOVERNED INPUTS
CANDIDATE_SOURCE_PATH = ROOT / "governed_candidate_adapter.py"
DECLARED_CANDIDATE_SHA256 = "REPLACE_WITH_FULL_SHA256"
CANDIDATE_IMPLEMENTATION_ID = "REPLACE_WITH_GOVERNED_IMPLEMENTATION_ID"

FORMAL_ORACLE_SPEC_PATH = ROOT / "RT_D_E_INDEPENDENT_FORMAL_ORACLE_SPEC_20260731_001.json"
DECLARED_SPEC_FILE_SHA256 = "REPLACE_WITH_FULL_SHA256"

CLEAN_ROOM_REVIEW = {
    "review_id": "REPLACE_WITH_REVIEW_ID",
    "reviewer": "REPLACE_WITH_REVIEWER",
    "reviewed_at": "REPLACE_WITH_ISO8601_TIMESTAMP",
    "candidate_code_not_consulted": False,
    "prior_notebook_code_not_consulted": False,
    "oracle_derived_only_from_frozen_spec": False,
    "signed_statement": "REPLACE_WITH_SIGNED_STATEMENT_OR_GOVERNED_ATTESTATION"
}

ALLOW_AUTHORITATIVE_EXECUTION = False


In [ ]:

def governance_preflight():
    failures = []

    if not ALLOW_AUTHORITATIVE_EXECUTION:
        failures.append("ALLOW_AUTHORITATIVE_EXECUTION is false.")

    if not CANDIDATE_SOURCE_PATH.exists():
        failures.append(f"Candidate source missing: {CANDIDATE_SOURCE_PATH}")
    if not FORMAL_ORACLE_SPEC_PATH.exists():
        failures.append(f"Formal oracle specification missing: {FORMAL_ORACLE_SPEC_PATH}")

    placeholder_values = {
        "DECLARED_CANDIDATE_SHA256": DECLARED_CANDIDATE_SHA256,
        "CANDIDATE_IMPLEMENTATION_ID": CANDIDATE_IMPLEMENTATION_ID,
        "DECLARED_SPEC_FILE_SHA256": DECLARED_SPEC_FILE_SHA256,
        "review_id": CLEAN_ROOM_REVIEW.get("review_id"),
        "reviewer": CLEAN_ROOM_REVIEW.get("reviewer"),
        "reviewed_at": CLEAN_ROOM_REVIEW.get("reviewed_at"),
        "signed_statement": CLEAN_ROOM_REVIEW.get("signed_statement"),
    }
    for name, value in placeholder_values.items():
        if not value or "REPLACE_WITH" in str(value):
            failures.append(f"Placeholder unresolved: {name}")

    for flag in (
        "candidate_code_not_consulted",
        "prior_notebook_code_not_consulted",
        "oracle_derived_only_from_frozen_spec",
    ):
        if CLEAN_ROOM_REVIEW.get(flag) is not True:
            failures.append(f"Clean-room review flag not affirmed: {flag}")

    if CANDIDATE_SOURCE_PATH.exists() and "REPLACE_WITH" not in DECLARED_CANDIDATE_SHA256:
        actual = sha256_file(CANDIDATE_SOURCE_PATH)
        if actual != DECLARED_CANDIDATE_SHA256:
            failures.append(f"Candidate SHA-256 mismatch: declared={DECLARED_CANDIDATE_SHA256}, actual={actual}")

    if FORMAL_ORACLE_SPEC_PATH.exists() and "REPLACE_WITH" not in DECLARED_SPEC_FILE_SHA256:
        actual = sha256_file(FORMAL_ORACLE_SPEC_PATH)
        if actual != DECLARED_SPEC_FILE_SHA256:
            failures.append(f"Specification SHA-256 mismatch: declared={DECLARED_SPEC_FILE_SHA256}, actual={actual}")

    return failures

PREFLIGHT_FAILURES = governance_preflight()
if PREFLIGHT_FAILURES:
    raise RuntimeError(
        "AUTHORITATIVE EXECUTION BLOCKED:\n- " + "\n- ".join(PREFLIGHT_FAILURES)
    )

print("Governance preflight passed.")


In [ ]:

FORMAL_ORACLE_SPEC = json.loads(FORMAL_ORACLE_SPEC_PATH.read_text(encoding="utf-8"))

spec_capture = {
    "artifact_id": FORMAL_ORACLE_SPEC.get("artifact_id"),
    "declared_file_sha256": DECLARED_SPEC_FILE_SHA256,
    "verified_file_sha256": sha256_file(FORMAL_ORACLE_SPEC_PATH),
    "content_sha256": FORMAL_ORACLE_SPEC.get("content_sha256")
}
candidate_capture = {
    "implementation_id": CANDIDATE_IMPLEMENTATION_ID,
    "declared_sha256": DECLARED_CANDIDATE_SHA256,
    "verified_sha256": sha256_file(CANDIDATE_SOURCE_PATH),
    "source_name": CANDIDATE_SOURCE_PATH.name
}

write_json(OUT / "formal_oracle_spec_capture.json", spec_capture)
write_json(OUT / "candidate_provenance.json", candidate_capture)
write_json(OUT / "clean_room_review_signoff.json", CLEAN_ROOM_REVIEW)

spec_capture, candidate_capture


## Independent clean-room oracle

In [ ]:

THRESHOLD_ENV = {
    "alpha": 0.1375,
    "beta": 0.4125,
    "gamma": 0.6875,
    "delta": 0.9375
}
CONTEXTS = ["A1", "B2", "C3", "D4", "E5"]
SEEDS = [26031, 26039, 26047, 26051, 26053]

def cr_same_context(a, b):
    return isinstance(a, str) and isinstance(b, str) and bool(a) and a == b

def cr_exact_bind(witness, payload):
    return isinstance(witness, dict) and witness.get("token") == payload

def cr_ordered_history(history):
    if not isinstance(history, list) or not history:
        return False
    if not all(isinstance(x, dict) and "step" in x and "state" in x for x in history):
        return False
    steps = [x["step"] for x in history]
    if not all(isinstance(s, int) and not isinstance(s, bool) for s in steps):
        return False
    return all(a < b for a, b in zip(steps, steps[1:]))

def cleanroom_representable(r, env):
    ordered = [
        ("REJECT_TYPE", r.get("relation_type") == "SourceRelation"),
        ("REJECT_CONTEXT", cr_same_context(r.get("context"), r.get("target_context"))),
        ("REJECT_PROFILE", r.get("profile") in env),
        ("REJECT_WITNESS", isinstance(r.get("witness"), dict)),
        ("REJECT_WITNESS", cr_exact_bind(r.get("witness"), r.get("source_payload"))),
        ("REJECT_HISTORY", isinstance(r.get("history"), list) and bool(r.get("history"))),
        ("REJECT_HISTORY", cr_ordered_history(r.get("history"))),
        ("REJECT_HISTORY", isinstance(r.get("history"), list) and bool(r.get("history")) and r["history"][-1].get("state") == r.get("target")),
    ]
    for rejection, passed in ordered:
        if not passed:
            return rejection
    return "REPRESENTABLE"

def cleanroom_noncollapsed(r, env):
    profile = r.get("profile")
    if profile not in env:
        return "REJECT_PROFILE"
    value = r.get("distinction")
    if not isinstance(value, (int, float)) or isinstance(value, bool) or value <= 0:
        return "REJECT_DISTINCTION"
    if value <= env[profile]:
        return "REJECT_SUBTHRESHOLD"
    return "NON_COLLAPSED"

def cleanroom_admissible(r, env):
    return (
        cleanroom_representable(r, env) == "REPRESENTABLE"
        and cleanroom_noncollapsed(r, env) == "NON_COLLAPSED"
    )


## Load frozen governed candidate

In [ ]:

module_spec = importlib.util.spec_from_file_location("governed_candidate", CANDIDATE_SOURCE_PATH)
if module_spec is None or module_spec.loader is None:
    raise RuntimeError("Unable to create import specification for governed candidate.")

governed_candidate = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(governed_candidate)

required_exports = [
    "candidate_representable",
    "candidate_noncollapsed",
    "candidate_admissible"
]
missing = [name for name in required_exports if not callable(getattr(governed_candidate, name, None))]
if missing:
    raise RuntimeError(f"Governed candidate is missing required callable exports: {missing}")

candidate_representable = governed_candidate.candidate_representable
candidate_noncollapsed = governed_candidate.candidate_noncollapsed
candidate_admissible = governed_candidate.candidate_admissible

print("Governed candidate loaded:", CANDIDATE_IMPLEMENTATION_ID)


## New frozen authoritative corpus

In [ ]:

def make_baseline(seed, context, profile, ordinal):
    rng = random.Random(f"N26-AUTH:{seed}:{context}:{profile}:{ordinal}")
    payload = {
        "head": rng.randint(100, 9999),
        "body": {
            "x": rng.randint(1, 100),
            "y": [rng.randint(1, 20), rng.randint(21, 40)]
        }
    }
    target = f"T_{seed}_{context}_{ordinal}"
    threshold = THRESHOLD_ENV[profile]
    return {
        "row_id": f"N26A_{seed}_{context}_{profile}_{ordinal}",
        "relation_type": "SourceRelation",
        "context": context,
        "target_context": context,
        "source_payload": payload,
        "witness": {"token": copy.deepcopy(payload), "authority": "frozen-corpus"},
        "history": [
            {"step": 7, "state": f"S0_{ordinal}"},
            {"step": 19, "state": f"S1_{ordinal}"},
            {"step": 43, "state": target},
        ],
        "target": target,
        "profile": profile,
        "distinction": round(threshold + 0.001 + rng.random() * 0.075, 12),
        "metadata": {"seed": seed, "ordinal": ordinal},
        "family": "baseline"
    }

BASELINES = [
    make_baseline(seed, context, profile, ordinal)
    for seed in SEEDS
    for context in CONTEXTS
    for profile in THRESHOLD_ENV
    for ordinal in range(3)
]

FAULTS = [
    "wrong_type",
    "empty_context",
    "cross_context",
    "unknown_profile",
    "missing_witness",
    "witness_scalar",
    "witness_corruption",
    "missing_history",
    "empty_history",
    "malformed_history_item",
    "boolean_history_step",
    "duplicate_history_step",
    "descending_history",
    "false_terminal",
    "nonnumeric_distinction",
    "boolean_distinction",
    "zero_distinction",
    "negative_distinction",
    "threshold_minus_epsilon",
    "threshold_exact",
    "threshold_plus_epsilon",
    "metadata_enrichment",
    "combined_type_context_fault",
    "combined_context_profile_fault",
    "combined_all_faults"
]

def mutate(row, family):
    r = copy.deepcopy(row)
    r["row_id"] = f"{row['row_id']}__{family}"
    r["parent_id"] = row["row_id"]
    r["family"] = family

    if family == "wrong_type":
        r["relation_type"] = "DerivedRelation"
    elif family == "empty_context":
        r["context"] = ""
        r["target_context"] = ""
    elif family == "cross_context":
        r["target_context"] = next(c for c in CONTEXTS if c != r["context"])
    elif family == "unknown_profile":
        r["profile"] = "unregistered"
    elif family == "missing_witness":
        r["witness"] = None
    elif family == "witness_scalar":
        r["witness"] = 17
    elif family == "witness_corruption":
        r["witness"]["token"]["body"]["x"] += 1
    elif family == "missing_history":
        r.pop("history", None)
    elif family == "empty_history":
        r["history"] = []
    elif family == "malformed_history_item":
        r["history"][1] = {"step": 19}
    elif family == "boolean_history_step":
        r["history"][1]["step"] = True
    elif family == "duplicate_history_step":
        r["history"][1]["step"] = r["history"][0]["step"]
    elif family == "descending_history":
        r["history"] = list(reversed(r["history"]))
    elif family == "false_terminal":
        r["history"][-1]["state"] = "FALSE_TERMINAL"
    elif family == "nonnumeric_distinction":
        r["distinction"] = "0.9"
    elif family == "boolean_distinction":
        r["distinction"] = True
    elif family == "zero_distinction":
        r["distinction"] = 0
    elif family == "negative_distinction":
        r["distinction"] = -0.0001
    elif family == "threshold_minus_epsilon":
        r["distinction"] = THRESHOLD_ENV[r["profile"]] - 1e-12
    elif family == "threshold_exact":
        r["distinction"] = THRESHOLD_ENV[r["profile"]]
    elif family == "threshold_plus_epsilon":
        r["distinction"] = THRESHOLD_ENV[r["profile"]] + 1e-12
    elif family == "metadata_enrichment":
        r["metadata"]["irrelevant"] = {"a": [1, 2, 3], "b": "ignored"}
    elif family == "combined_type_context_fault":
        r["relation_type"] = "DerivedRelation"
        r["target_context"] = "OTHER"
    elif family == "combined_context_profile_fault":
        r["target_context"] = "OTHER"
        r["profile"] = "unregistered"
    elif family == "combined_all_faults":
        r["relation_type"] = "DerivedRelation"
        r["target_context"] = "OTHER"
        r["profile"] = "unregistered"
        r["witness"] = None
        r["history"] = []
        r["distinction"] = -1

    return r

ROWS = []
for baseline in BASELINES:
    ROWS.append(copy.deepcopy(baseline))
    ROWS.extend(mutate(baseline, family) for family in FAULTS)

CORPUS_SHA256 = sha256_bytes(canonical(ROWS).encode("utf-8"))
write_jsonl(OUT / "authoritative_comparison_corpus.jsonl", ROWS)
write_json(OUT / "corpus_capture.json", {
    "baselines": len(BASELINES),
    "fault_families": len(FAULTS),
    "rows_per_pass": len(ROWS),
    "corpus_sha256": CORPUS_SHA256,
    "seeds": SEEDS,
    "contexts": CONTEXTS,
    "threshold_environment": THRESHOLD_ENV
})

print("Baselines:", len(BASELINES))
print("Fault families:", len(FAULTS))
print("Rows per pass:", len(ROWS))
print("Corpus SHA-256:", CORPUS_SHA256)


In [ ]:

def evaluate_candidate(row):
    return {
        "row_id": row["row_id"],
        "representable": candidate_representable(copy.deepcopy(row), copy.deepcopy(THRESHOLD_ENV)),
        "noncollapsed": candidate_noncollapsed(copy.deepcopy(row), copy.deepcopy(THRESHOLD_ENV)),
        "admissible": candidate_admissible(copy.deepcopy(row), copy.deepcopy(THRESHOLD_ENV)),
    }

def evaluate_cleanroom(row):
    return {
        "row_id": row["row_id"],
        "representable": cleanroom_representable(copy.deepcopy(row), copy.deepcopy(THRESHOLD_ENV)),
        "noncollapsed": cleanroom_noncollapsed(copy.deepcopy(row), copy.deepcopy(THRESHOLD_ENV)),
        "admissible": cleanroom_admissible(copy.deepcopy(row), copy.deepcopy(THRESHOLD_ENV)),
    }

def compare(row):
    candidate = evaluate_candidate(row)
    oracle = evaluate_cleanroom(row)
    fields = [
        field for field in ("representable", "noncollapsed", "admissible")
        if candidate[field] != oracle[field]
    ]
    return {
        "row_id": row["row_id"],
        "parent_id": row.get("parent_id"),
        "family": row["family"],
        "candidate": candidate,
        "cleanroom": oracle,
        "agreement": not fields,
        "mismatch_fields": fields,
        "input_sha256": sha256_bytes(canonical(row).encode("utf-8"))
    }

PASS1 = [compare(row) for row in ROWS]

shuffled = copy.deepcopy(ROWS)
random.Random(26022602).shuffle(shuffled)
PASS2 = [compare(row) for row in shuffled]

def normalized_digest(results):
    return sha256_bytes(canonical(sorted(results, key=lambda x: x["row_id"])).encode("utf-8"))

PASS1_DIGEST = normalized_digest(PASS1)
PASS2_DIGEST = normalized_digest(PASS2)
REPLAY_AGREEMENT = PASS1_DIGEST == PASS2_DIGEST
COUNTEREXAMPLES = [row for row in PASS1 if not row["agreement"]]

print("Counterexamples:", len(COUNTEREXAMPLES))
print("Replay agreement:", REPLAY_AGREEMENT)


In [ ]:

LOOKUP = {row["row_id"]: row for row in PASS1}
INVARIANTS = []

for baseline in BASELINES:
    bid = baseline["row_id"]

    INVARIANTS.append({
        "id": "baseline_cleanroom_admission",
        "row_id": bid,
        "pass": (
            LOOKUP[bid]["cleanroom"]["representable"] == "REPRESENTABLE"
            and LOOKUP[bid]["cleanroom"]["noncollapsed"] == "NON_COLLAPSED"
            and LOOKUP[bid]["cleanroom"]["admissible"] is True
        )
    })

    expected_rep = {
        "wrong_type": "REJECT_TYPE",
        "empty_context": "REJECT_CONTEXT",
        "cross_context": "REJECT_CONTEXT",
        "unknown_profile": "REJECT_PROFILE",
        "missing_witness": "REJECT_WITNESS",
        "witness_scalar": "REJECT_WITNESS",
        "witness_corruption": "REJECT_WITNESS",
        "missing_history": "REJECT_HISTORY",
        "empty_history": "REJECT_HISTORY",
        "malformed_history_item": "REJECT_HISTORY",
        "boolean_history_step": "REJECT_HISTORY",
        "duplicate_history_step": "REJECT_HISTORY",
        "descending_history": "REJECT_HISTORY",
        "false_terminal": "REJECT_HISTORY",
        "combined_type_context_fault": "REJECT_TYPE",
        "combined_context_profile_fault": "REJECT_CONTEXT",
        "combined_all_faults": "REJECT_TYPE",
    }

    for family, expected in expected_rep.items():
        rid = f"{bid}__{family}"
        INVARIANTS.append({
            "id": "representability_rejection_precedence",
            "row_id": rid,
            "expected": expected,
            "pass": LOOKUP[rid]["cleanroom"]["representable"] == expected
        })

    for family in ("threshold_minus_epsilon", "threshold_exact"):
        rid = f"{bid}__{family}"
        INVARIANTS.append({
            "id": "strict_threshold_rejection",
            "row_id": rid,
            "pass": LOOKUP[rid]["cleanroom"]["noncollapsed"] == "REJECT_SUBTHRESHOLD"
        })

    rid = f"{bid}__threshold_plus_epsilon"
    INVARIANTS.append({
        "id": "strict_threshold_admission",
        "row_id": rid,
        "pass": LOOKUP[rid]["cleanroom"]["noncollapsed"] == "NON_COLLAPSED"
    })

    rid = f"{bid}__metadata_enrichment"
    INVARIANTS.append({
        "id": "metadata_enrichment_invariance",
        "row_id": rid,
        "pass": (
            LOOKUP[rid]["cleanroom"]["representable"] == LOOKUP[bid]["cleanroom"]["representable"]
            and LOOKUP[rid]["cleanroom"]["noncollapsed"] == LOOKUP[bid]["cleanroom"]["noncollapsed"]
            and LOOKUP[rid]["cleanroom"]["admissible"] == LOOKUP[bid]["cleanroom"]["admissible"]
        )
    })

INVARIANT_FAILURES = [item for item in INVARIANTS if not item["pass"]]

print("Invariant checks:", len(INVARIANTS))
print("Invariant failures:", len(INVARIANT_FAILURES))


In [ ]:

if COUNTEREXAMPLES or INVARIANT_FAILURES or not REPLAY_AGREEMENT:
    OUTCOME = "FAIL_COUNTEREXAMPLE_OR_INVARIANT_FAILURE"
else:
    OUTCOME = "PASS_BOUNDED_EQUIVALENCE"

summary = {
    "notebook_id": NOTEBOOK_ID,
    "result_id": RESULT_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "mode": "GOVERNED_FROZEN_CANDIDATE",
    "candidate_implementation_id": CANDIDATE_IMPLEMENTATION_ID,
    "candidate_sha256": DECLARED_CANDIDATE_SHA256,
    "specification_sha256": DECLARED_SPEC_FILE_SHA256,
    "clean_room_review_id": CLEAN_ROOM_REVIEW["review_id"],
    "claim_ceiling": "C2_LIMITATION_OR_NEGATIVE_RESULT",
    "proof_status": "NOT_ATTEMPTED",
    "counts": {
        "baselines": len(BASELINES),
        "fault_families": len(FAULTS),
        "rows_per_pass": len(ROWS),
        "total_evaluations": len(ROWS) * 2,
        "counterexamples": len(COUNTEREXAMPLES),
        "invariant_checks": len(INVARIANTS),
        "invariant_failures": len(INVARIANT_FAILURES)
    },
    "replay": {
        "pass1_digest": PASS1_DIGEST,
        "pass2_digest": PASS2_DIGEST,
        "agreement": REPLAY_AGREEMENT
    },
    "corpus_sha256": CORPUS_SHA256,
    "outcome": OUTCOME,
    "allowed_interpretation": "Bounded observed agreement only within the frozen finite comparison corpus.",
    "blocked_interpretations": [
        "Universal equivalence",
        "Formal equivalence proof",
        "Theorem promotion",
        "Claim promotion above C2",
        "Suppression of preserved disagreements"
    ]
}

falsification = {
    "result_id": RESULT_ID,
    "outcome": OUTCOME,
    "counterexamples": COUNTEREXAMPLES,
    "invariant_failures": INVARIANT_FAILURES,
    "replay_agreement": REPLAY_AGREEMENT,
    "preservation_rule": "Every mismatch and invariant failure is retained in full."
}

write_jsonl(OUT / "candidate_outputs.jsonl", [evaluate_candidate(row) for row in ROWS])
write_jsonl(OUT / "cleanroom_outputs.jsonl", [evaluate_cleanroom(row) for row in ROWS])
write_jsonl(OUT / "comparison_rows.jsonl", PASS1)
write_jsonl(OUT / "comparison_rows_replay_shuffled.jsonl", PASS2)
write_jsonl(OUT / "invariant_checks.jsonl", INVARIANTS)
write_json(OUT / "comparison_summary.json", summary)
write_json(OUT / "counterexamples.json", falsification)

summary


In [ ]:

artifact_files = sorted(
    path for path in OUT.iterdir()
    if path.is_file() and path.name not in {"manifest.json", f"{RESULT_ID}.zip"}
)

manifest = {
    "notebook_id": NOTEBOOK_ID,
    "result_id": RESULT_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "mode": "GOVERNED_FROZEN_CANDIDATE",
    "candidate_implementation_id": CANDIDATE_IMPLEMENTATION_ID,
    "candidate_sha256": DECLARED_CANDIDATE_SHA256,
    "specification_sha256": DECLARED_SPEC_FILE_SHA256,
    "clean_room_review": CLEAN_ROOM_REVIEW,
    "runtime": {
        "python": sys.version,
        "platform": platform.platform()
    },
    "artifacts": [
        {
            "name": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        }
        for path in artifact_files
    ],
    "outcome": OUTCOME,
    "claim_ceiling": "C2_LIMITATION_OR_NEGATIVE_RESULT"
}
write_json(OUT / "manifest.json", manifest)

archive = OUT / f"{RESULT_ID}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        if path.is_file() and path != archive:
            bundle.write(path, arcname=path.name)

print("Archive:", archive.resolve())
print("Archive SHA-256:", sha256_file(archive))


## Interpretation boundary

A successful execution may report:

```text
PASS_BOUNDED_EQUIVALENCE
```

This means only that no disagreement was observed within the frozen finite corpus under the verified governed inputs.

It does not establish universal equivalence or discharge formal proof obligations.
